### The Personal Knowledge Worker

In this week's challenge, I've built a personal knowledge worker that can answer questions about my personal information.

The knowledge worker is built using the following components:
- Langchain MarkdownTextSplitter
- LLM-Based Document Chunking
- OpenAI Embeddings & Chroma
- LLM Question Rewriter
- Merging Retrieved Chunks
- Gradio Chat Interface with Retrieved Chunks

In [1]:
from pathlib import Path
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from openai import OpenAI
from chromadb import PersistentClient 
from langchain_text_splitters import MarkdownTextSplitter
import os
import gradio as gr

load_dotenv(override=True)

if (os.getenv("OPENAI_API_KEY") is None):
    raise ValueError("OPENAI_API_KEY environment variable is not set.")

openai = OpenAI()

MODEL = "gpt-4.1-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
RETRIEVAL_K = 5
CHUNK_SIZE = 100
KNOWLEDGE_BASE_PATH = Path().cwd() / "knowledge_base"
DB_NAME = str(Path.cwd() / "vector_db")
COLLECTION_NAME = "personal_info"

chroma = PersistentClient(path=DB_NAME)

In [2]:
class Chunk(BaseModel):
  headline: str = Field("A few words of title that describes the contents of the chunk")
  text: str = Field("The original text of this chunk that should be kept as is, shouldn't be modified in any way")

  def as_result(self):
    return self.headline + "\n\n" + self.text

class Chunks(BaseModel):
  chunks: list[Chunk]

class RewrittenQuestion(BaseModel):
  question: str = Field("A modified query of the user's question that is the most likely to retrieve the most relevant results from the vector dataset")

In [3]:
def ai_document_splitter(document: str):
  system_message = """
    You are an expert document splitter for perfect RAG results. 
  """

  user_message = f"""
    You take a document and you split the document into overlapping chunks for a personal knowledge base.
    The document contains personal information of a specific person.
    A chatbot will use these chunks to answer questions about the person.
    You should divide up the document as you see fit, being sure that the entire document is returned across the chunks - don't leave anything out.
    This document should ALWAYS be split into at about 50 chunks, but you can have more or less as appropriate, ensuring that there are individual chunks to answer specific questions.
    There should be overlap between the chunks as appropriate; so you have the same text in multiple chunks for best retrieval results.
    For each chunk, you should provide a headline, a summary, and the original text of the chunk.
    Together your chunks should represent the entire document with overlap.
    Here is the document:
    
    {document}

    Respond with about 50 chunks!
  """
  response_json = openai.responses.parse(
    model=MODEL,
    input=[{"role": "system", "content": system_message}, {"role": "user", "content": user_message}],
    text_format=Chunks
  )
  response = response_json.output_parsed.chunks
  print(len(response), print(response[0].as_result()))

  return [chunk.as_result() for chunk in response]

In [4]:
def langchain_text_splitter(document: str):
  text_splitter = MarkdownTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=10)
  chunks = text_splitter.split_text(document)
  return chunks

In [5]:
def vectorize_chunks(chunks: list[str]):
  if COLLECTION_NAME in [c.name for c in chroma.list_collections()]:
    chroma.delete_collection(name=COLLECTION_NAME)

  embeddings = openai.embeddings.create(
    input=chunks,
    model=EMBEDDING_MODEL
  ).data

  vectors = [embedding.embedding for embedding in embeddings]
  collection = chroma.get_or_create_collection(name=COLLECTION_NAME)
  ids = [str(i) for i in range(len(chunks))]
  collection.add(ids=ids, embeddings=vectors, documents=chunks)
  print(f"Vectorstore created with {collection.count()} documents")

In [6]:
def query(question):
  vectorized_question = openai.embeddings.create(model=EMBEDDING_MODEL, input=[question]).data[0].embedding
  collection = chroma.get_collection(name=COLLECTION_NAME)
  results = collection.query(query_embeddings=[vectorized_question], n_results=RETRIEVAL_K)
  result_chunks = []
  for result in results['documents']:
    result_chunks.append(result)
  return result

In [7]:
def read_document():
  document = []
  for file in KNOWLEDGE_BASE_PATH.glob("*.md"):
    with open(file, "r", encoding="utf-8") as f:
      document.append(f.read())
  return document[0]

In [8]:
def ingest(is_ai: bool):
  document = read_document()
  chunks = []
  if (is_ai):
    chunks = ai_document_splitter(document)
  else:
    chunks = langchain_text_splitter(document)
  vectorize_chunks(chunks)

In [69]:
ingest(is_ai=False)

Vectorstore created with 39 documents


In [9]:
def question_rewriter(question: str, history: list[dict] = []):
  system_prompt = "You are an expert at rewriting user queries for perfect RAG results"
  user_prompt = f"""
    Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base.
    You are in a conversation with a user, answering questions about his personal information.
    You are about to look up information in a Knowledge Base to answer the user's question.

    This is the history of your conversation so far with the user:
    {history}

    And this is the user's current question:
    {question}

    Respond only with a short, refined question that you will use to search the Knowledge Base.
    It should be a VERY short specific question most likely to surface content. Focus on the question details.
    IMPORTANT: Respond ONLY with the precise knowledgebase query, nothing else.
  """

  response = openai.responses.parse(
    model=MODEL,
    text_format=RewrittenQuestion,
    input=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
  )
  rewritten_question = response.output_parsed.question
  return rewritten_question

In [10]:
def merge_results(original_results, rewritten_question_results):
  merged = set(original_results + rewritten_question_results)
  return list(merged)

In [11]:
def make_rag_messages(chunks, question, history):
  system_prompt = f"""
  You are Muse, a loyal assistant of a good man Lythmass.
  Your job is to assist him achieve his goals in different fields: coding, weight-lifting, language-learning, essay-writing, sports, dreams, etc...
  If there's something you don't know, just say so.
  Here's some additional context that might help you: \n
  """
  chunks = ",\n".join(chunk for chunk in chunks)
  system_prompt += chunks

  return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

  

In [12]:
def answer_question(history: list[dict] = []):

  if not history:
        return history, "No chunks retrieved yet."
    
  question = history[-1]["content"]
  past_history = history[:-1]

  chunks = query(question)
  rewritten_question = question_rewriter(question, past_history)
  rewritten_question_chunks = query(rewritten_question)
  merged_chunks = merge_results(chunks, rewritten_question_chunks)

  chunks_text = "### 📚 Retrieved Context Chunks\n\n"
  for i, chunk in enumerate(merged_chunks):
    chunks_text += f"**Chunk {i+1}:**\n{chunk}\n\n---\n\n"


  messages = make_rag_messages(chunks=merged_chunks, question=question, history=past_history)


  stream = openai.chat.completions.create(
    model=MODEL,
    messages=messages,
    stream=True
  )

  history.append({"role": "assistant", "content": ""})
  
  response = ""
  for chunk in stream:
      response += chunk.choices[0].delta.content or ''
      history[-1]["content"] = response
      
      yield history, chunks_text


In [16]:
def add_user_message(history: list[dict], question: str):
    """Appends the user's question to the chat history and clears the input box."""
    if question.strip() == "":
        return history, ""
    history.append({"role": "user", "content": question})
    return history, ""

with gr.Blocks(title="RAG Assistant") as demo:
    gr.Markdown("# 🤖 RAG Copilot with Source Context")
    
    with gr.Row():
        # Column 1: Chat Interface (Takes up 60% of the width)
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(type="messages", label="Chat History", height=500)
            with gr.Row():
                msg = gr.Textbox(
                    show_label=False,
                    placeholder="Ask me anything...",
                    container=False,
                    scale=4
                )
                submit_btn = gr.Button("Send", variant="primary", scale=1)
                
        # Column 2: Retrieved Chunks (Takes up 40% of the width)
        with gr.Column(scale=2):
            chunks_display = gr.Markdown(
                value="*Retrieved chunks will appear here once you submit a question.*",
                label="Retrieved Chunks"
            )
    
    submit_click = submit_btn.click(
        fn=add_user_message, 
        inputs=[chatbot, msg], 
        outputs=[chatbot, msg]
    ).then(
        fn=answer_question,
        inputs=[chatbot],
        outputs=[chatbot, chunks_display]
    )
    
    submit_enter = msg.submit(
        fn=add_user_message, 
        inputs=[chatbot, msg], 
        outputs=[chatbot, msg]
    ).then(
        fn=answer_question,
        inputs=[chatbot],
        outputs=[chatbot, chunks_display]
    )

demo.launch(inbrowser= True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
